In [8]:
import pandas as pan 
import xarray as xar
import numpy as np
import h5py as h5

In [2]:
with h5.File('HITRAN_o2_uncoupled_0-1750GHz.hdf5', 'r') as hf:
    o2_ds = xar.Dataset(
        {name: ('line', hf[name][:]) for name in hf.keys()},
        attrs=dict(hf.attrs),
    )

print(o2_ds)

<xarray.Dataset> Size: 34kB
Dimensions:              (line: 206)
Dimensions without coordinates: line
Data variables: (12/19)
    a                    (line) float64 2kB 1.013e-31 9.249e-32 ... 2.073e-08
    delta_air            (line) float64 2kB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    elower               (line) float64 2kB 4.383e+03 4.701e+03 ... 1.685e+03
    gamma_air            (line) float64 2kB 0.0346 0.0346 ... 0.0475 0.0459
    gamma_self           (line) float64 2kB 0.034 0.034 0.034 ... 0.047 0.045
    global_lower_quanta  (line) |S15 3kB b'X      0' b'X      0' ... b'X      1'
    ...                   ...
    local_lower_quanta   (line) |S15 3kB b'Q 55Q 55     g' ... b'S  9R  9     d'
    local_upper_quanta   (line) |S15 3kB b'' b'' b'' b'' b'' ... b'' b'' b'' b''
    molec_id             (line) int32 824B 7 7 7 7 7 7 7 7 7 ... 7 7 7 7 7 7 7 7
    n_air                (line) float64 2kB 0.72 0.72 0.72 ... 0.72 0.72 0.72
    nu                   (line) float64 2kB 1e-06 1e

In [5]:
o2_csv_df = pan.read_csv(
    'coupled_O2_line_data/oxygen_line_parameters.csv',
    skiprows=3, thousands=',', encoding='latin-1',
)
o2_csv_df.index.name = 'line'
o2_csv_ds = o2_csv_df.to_xarray()

print(o2_csv_ds)

<xarray.Dataset> Size: 4kB
Dimensions:                         (line: 38)
Coordinates:
  * line                            (line) int64 304B 0 1 2 3 4 ... 34 35 36 37
Data variables: (12/13)
    Quantum Transition (N±)         (line) object 304B '1-' '1+' ... '37-' '37+'
    Measured Freq. (MHz)            (line) object 304B '118750.340' ... 'Ñ'
    Calc. Freq. mi (GHz)            (line) float64 304B 118.8 56.26 ... 68.96
    c_O2 Broadening (MHz/Torr)      (line) object 304B '2.230' '2.238' ... 'Ñ'
    c_N2 Broadening (MHz/Torr)      (line) object 304B '2.245' '2.310' ... 'Ñ'
    Calculated c_air (MHz/Torr)     (line) float64 304B 2.242 2.295 ... nan nan
    ...                              ...
    Intensity a1 (kHz/kPa)          (line) float64 304B 940.3 543.4 ... 1.0 1.3
    Intensity Exp. a2               (line) float64 304B 0.01 0.014 ... 9.65
    Width Coeff. a3 (GHz/10^5 Pa)   (line) float64 304B 1.664 1.703 ... 0.669
    Mixing Coeff. a5 (10^-5 Pa^-1)  (line) float64 304B -0.04

In [ ]:
import scipy.constants as sc

# Keep original line indices so each partition retains source-row provenance.
o2_ds = o2_ds.assign_coords(line=np.arange(o2_ds.sizes['line']))
coupled_lines = o2_ds.isel(line=slice(0, 0)).copy()
uncoupled_lines = o2_ds.isel(line=slice(0, 0)).copy()

calc_freq_GHz = o2_csv_df['Calc. Freq. mi (GHz)'].dropna().to_numpy(dtype=float)
nu_GHz = o2_ds['nu'].to_numpy() * sc.c / 1e7  # cm^-1 to GHz
match_tolerance_GHz = 1e-4  # 0.1 MHz

coupled_indices = []
uncoupled_indices = []
for i, freq_GHz in enumerate(nu_GHz):
    if np.any(np.abs(calc_freq_GHz - freq_GHz) <= match_tolerance_GHz):
        coupled_indices.append(i)
    else:
        uncoupled_indices.append(i)

coupled_lines = o2_ds.isel(line=coupled_indices)
uncoupled_lines = o2_ds.isel(line=uncoupled_indices)

print(f'Coupled matches: {coupled_lines.sizes["line"]}')
print(f'Uncoupled lines: {uncoupled_lines.sizes["line"]}')
# print(coupled_lines)
# print(uncoupled_lines)

Coupled matches: 38
Uncoupled lines: 168


In [17]:
coupled_lines = coupled_lines.copy(deep=True)

calc_freq_all = pan.to_numeric(
    o2_csv_df['Calc. Freq. mi (GHz)'], errors='coerce'
).to_numpy(dtype=float)
valid_calc_rows = np.flatnonzero(np.isfinite(calc_freq_all))
calc_freq_valid = calc_freq_all[valid_calc_rows]

nu_updated = coupled_lines['nu'].to_numpy().copy()
gamma_self_updated = coupled_lines['gamma_self'].to_numpy().copy()
gamma_air_updated = coupled_lines['gamma_air'].to_numpy().copy()

mhz_to_cm_inv = 1e6 / (sc.c * 100.0)
mhz_per_torr_to_cm_inv_per_atm = mhz_to_cm_inv * 760.0

for j, nu_cm_inv in enumerate(nu_updated):
    freq_GHz = nu_cm_inv * sc.c / 1e7
    nearest = int(np.argmin(np.abs(calc_freq_valid - freq_GHz)))
    csv_row = o2_csv_df.iloc[valid_calc_rows[nearest]]

    if np.abs(calc_freq_valid[nearest] - freq_GHz) > match_tolerance_GHz:
        raise ValueError(f'No CSV frequency match within tolerance for line {j}')

    measured_MHz = pan.to_numeric(csv_row['Measured Freq. (MHz)'], errors='coerce')
    if np.isfinite(measured_MHz):
        nu_updated[j] = float(measured_MHz) * mhz_to_cm_inv

    broadening_o2 = pan.to_numeric(csv_row['c_O2 Broadening (MHz/Torr)'], errors='coerce')
    if np.isfinite(broadening_o2):
        gamma_self_updated[j] = float(broadening_o2) * mhz_per_torr_to_cm_inv_per_atm

    broadening_air = pan.to_numeric(csv_row['Calculated c_air (MHz/Torr)'], errors='coerce')
    if np.isfinite(broadening_air):
        gamma_air_updated[j] = float(broadening_air) * mhz_per_torr_to_cm_inv_per_atm

for name, values in (
    ('nu', nu_updated),
    ('gamma_self', gamma_self_updated),
    ('gamma_air', gamma_air_updated),
):
    attrs = dict(coupled_lines[name].attrs)
    coupled_lines[name] = ('line', values)
    coupled_lines[name].attrs = attrs

coupled_lines['n_air'] = ('line', np.full(coupled_lines.sizes['line'], 0.8))

print(f'Updated {coupled_lines.sizes["line"]} coupled O2 lines.')

Updated 38 coupled O2 lines.


In [ ]:
am_nonresonant_broad = {  "freq": 0.000030,           # line frequency                     [GHz]
                    "S": 8.993774e-36,          # line intensity              [cm^2 * GHz]
                    "Elo": -1.438769,           # lower-state energy                   [K]
                    "gam_air": 5.5920e-04,      # air-broadening coefficient  [GHz / mbar]
                    "gam_self": 5.5920e-04,     # self-broadening coefficient [GHz / mbar] 
                    "n_air": 0.80,              # T dependence exponent of gam_air
                    "delta_air": 0.0000e+00,    # air-induced pressure shift  [GHz / mbar]  
                    "iso": 1                    # HITRAN isotopologue number
                 }

am_nonresonant_mix = {  "y0": 0.000e+00,        # first-order mixing coefficient
                        "y1": 0.000e+00,        # temperature dependence linear parameter
                        "yx": 0.80,             # temperature dependence exponent
                        "g0": 0.000e+00,        # second-order mixing coefficient on intensity
                        "g1": 0.000e+00,        # temperature dependence linear parameter
                        "gx": 1.60,             # temperature dependence exponent
                        "d0": 0.0000e+00,       # second-order mixing coefficient on line frequency
                        "dx": 0.000e+00,        # temperature dependence exponent
                        "T_ref": 300.0          # reference temperature
                      }

